<a href="https://colab.research.google.com/github/Ololade117/Iroko/blob/main/Iroko_Bot_finetuned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1: Install Required Libraries

First, we need to install the necessary libraries for fine-tuning. This includes `transformers` for model handling, `datasets` for data loading, `peft` for parameter-efficient fine-tuning, `accelerate` for distributed training, and `bitsandbytes` for 8-bit optimization.

In [1]:
%%capture
# Upgrade to latest versions to resolve the top_k_top_p_filtering ImportError
!pip install -U accelerate peft bitsandbytes transformers trl datasets tokenizers

### Step 2.1: Authenticate with Hugging Face

To access gated models like `google/gemma-2b-it`, you need to authenticate with your Hugging Face token. Make sure you have accepted the model's terms of use on its Hugging Face page.

In [2]:
from huggingface_hub import login

# You will be prompted to enter your Hugging Face token
login()

### Step 2: Load and Prepare the Dataset

We will download the `train.jsonl` dataset and then load it using the `datasets` library.

In [3]:
import requests
from datasets import load_dataset
import pandas as pd

dataset_url = "https://github.com/Ololade117/Iroko/raw/main/dataset/train.jsonl"
local_file_name = "train.jsonl"

# Download the dataset
print(f"Downloading dataset from {dataset_url}...")
response = requests.get(dataset_url, stream=True)
response.raise_for_status() # Raise an exception for HTTP errors

with open(local_file_name, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)
print(f"Dataset downloaded to {local_file_name}")

# Load the dataset using datasets library
dataset = load_dataset('json', data_files=local_file_name)

# Display the first few examples
print("\nFirst 5 examples from the dataset:")
for i in range(min(5, len(dataset['train']))):
    print(dataset['train'][i])

# Convert to pandas DataFrame for easier inspection if needed
df = pd.DataFrame(dataset['train'])
print(f"\nDataset loaded into a pandas DataFrame with {len(df)} rows and {len(df.columns)} columns.")
print("DataFrame Info:")
df.info()

Dataset downloaded to train.jsonl


Generating train split: 0 examples [00:00, ? examples/s]


First 5 examples from the dataset:
{'instruction': "I feel like I could never be with anyone because no one would want me: What do I do if I have been feeling like I could never be with anyone because no one would want me. Or I couldn't have many friends because of who I am. It's strange I want to be loved but I'd hate to be because I always lose.", 'input': '', 'output': 'What would make you feel no one wants to be with you?'}
{'instruction': "When I'm in large crowds I get angry and I just can't deal with people. I don't really like other people (I prefer animals) they make me nervous and scared. I lay awake at night thinking and having conversations in my head and i almost always end up making myself feel terrible and crying, I have more conversions in my head than I do with actual people. I don't know what's wrong with me and why I feel this way. What should I do?", 'input': '', 'output': "Reaching out to talk about these issues is an important first step. Finding professional ser

### Step 3: Prepare the Dataset for Training

Before fine-tuning, the text data needs to be tokenized and formatted appropriately for the model. We'll load the tokenizer for the base model and define a preprocessing function.

In [4]:
from transformers import AutoTokenizer
import os
from google.colab import userdata

# Define the model ID
model_id = "google/medgemma-1.5-4b-it"

# Try to get token from Colab Secrets
try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ["HF_TOKEN"] = hf_token
except Exception:
    print("Warning: HF_TOKEN not found in Secrets. Please add it to access MedGemma.")

# Load the tokenizer
# We use use_fast=False as a fallback for the 'ModelWrapper' error
print(f"Loading tokenizer for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True,
    use_fast=False
)
tokenizer.pad_token = tokenizer.eos_token

# Define a formatting function for the dataset
def formatting_func(examples):
    outputs = []
    # Iterate through the batch
    for i in range(len(examples['instruction'])):
        instruction = examples["instruction"][i]
        response = examples.get("output", [""])[i]

        if instruction and response:
            text = f"### Human: {instruction}\n### Assistant: {response}{tokenizer.eos_token}"
        else:
            # Fallback if one field is empty
            text = (instruction or "") + (response or "") + tokenizer.eos_token
        outputs.append(text)
    return {"text": outputs}

# Apply formatting
print("Formatting dataset...")
processed_dataset = dataset['train'].map(formatting_func, batched=True)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

# Tokenize
print("Tokenizing dataset...")
tokenized_dataset = processed_dataset.map(tokenize_function, batched=True)

# Keep only necessary columns for training
tokenized_dataset = tokenized_dataset.remove_columns([col for col in tokenized_dataset.column_names if col not in ['input_ids', 'attention_mask']])

print(f"\nPrepared dataset has {len(tokenized_dataset)} examples.")
print("Example decoded:")
print(tokenizer.decode(tokenized_dataset[0]['input_ids']))

Loading tokenizer for google/medgemma-1.5-4b-it...
Formatting dataset...


Map:   0%|          | 0/2309 [00:00<?, ? examples/s]

Tokenizing dataset...


Map:   0%|          | 0/2309 [00:00<?, ? examples/s]


Prepared dataset has 2309 examples.
Example decoded:
<bos>### Human: I feel like I could never be with anyone because no one would want me: What do I do if I have been feeling like I could never be with anyone because no one would want me. Or I couldn't have many friends because of who I am. It's strange I want to be loved but I'd hate to be because I always lose.
### Assistant: What would make you feel no one wants to be with you?<eos>


### Step 4: Configure and Run Fine-tuning

This step involves loading the base model, configuring LoRA (Low-Rank Adaptation) for efficient fine-tuning, defining training arguments, and then using the `SFTTrainer` to train the model.

In [5]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer

# 1. Load the base model with 4-bit quantization
print("\nLoading base model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map="auto"
)

# Prepare model for 4-bit training
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

# 2. Configure LoRA
print("Configuring LoRA...")
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

# 3. Define TrainingArguments
print("Defining Training Arguments...")
training_arguments = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    optim="paged_adamw_32bit",
    save_steps=100,
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    warmup_steps=100,
    lr_scheduler_type="constant",
    report_to="none",
    gradient_checkpointing=True
)

# 4. Set up SFTTrainer
# Note: 'max_seq_length' and 'packing' removed as the dataset is already tokenized
print("Setting up SFTTrainer...")
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=training_arguments
)

# 5. Start training
print("\nStarting fine-tuning...")
trainer.train()

# Save the fine-tuned model
output_dir = "./medgemma-finetuned-iroko"
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Fine-tuned model saved to {output_dir}")


Loading base model...


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Configuring LoRA...
Defining Training Arguments...
Setting up SFTTrainer...


Building labels for train dataset:   0%|          | 0/2309 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2309 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/2309 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 1}.



Starting fine-tuning...


Step,Training Loss


KeyboardInterrupt: 